#### C. Implementation of the RSA function and its inverse

In [320]:
import os                                                   # For navegation in the file folders.
import secrets                                              # To use entropy for true randomness.
from aux_func import *                                     # Auxiliar functions for the project.
from cryptography.hazmat.primitives.asymmetric import rsa   # For generating keys.
from cryptography.hazmat.primitives import hashes           # Function H(n, r).

We start by creating the functions that will mask and unmask our private key. 

In [321]:
def rsa_trapdoor_permutation(public_key, r_int):            # Pure RSA, with the private key, it takes it's public numbers and calculate the R.
    e = public_key.public_numbers().e
    n = public_key.public_numbers().n
    return pow(r_int, e, n)

def rsa_inverse_trapdoor(private_key, c_r):                 # Inverse function of the one above.
    d = private_key.private_numbers().d
    n = private_key.private_numbers().public_numbers.n
    return pow(c_r, d, n)

In [322]:
def custom_encrypt(arguments):                              # Hybrid encription is done here.
    public_key, message = arguments[0], arguments[1]
    n = public_key.public_numbers().n
    r = secrets.randbelow(n)
    rsa_r = rsa_trapdoor_permutation(public_key, r)
    rsa_r_bytes = rsa_r.to_bytes(256, 'big')                # 2048 bits = 256 bytes
    ciphertext_blocks = [rsa_r_bytes]

    r_bytes = r.to_bytes(256, 'big')
    l_size = 32                                             # SHA256 output size

    for i in range(0, len(message), l_size):                # Hashing the message or file.
        block_idx = i // l_size
        chunk = message[i : i + l_size]
        
        digest = hashes.Hash(hashes.SHA256())
        
        hash_update_time = measure_performance_time(digest.update, block_idx.to_bytes(4, 'big') + r_bytes)[0]
        hash_finalize_time, block_with_hash = measure_performance_time(digest.finalize)
        hash_time = hash_update_time + hash_finalize_time
        
        ciphertext_blocks.append(xor_bytes(chunk, block_with_hash))

    return b"".join(ciphertext_blocks), hash_time

In [323]:
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

In [ ]:
base_dir = os.getcwd()
target_dir = os.path.join(base_dir, "03 - data", "inputs by size")

all_files, folder_path, file_size = get_files(base_dir)
total_time = 0

print(f"\nProcessing {len(all_files)} files from folder '{file_size}'...")
print(f"{'File':<20} | {'Size':<10} | {'Hash Time (s)':<12} | {'RSA Time (s)':<12}")
print("-" * 65)

if (all_files):
    for file_name in all_files:
        file_path = os.path.join(folder_path, file_name)

        with open(file_path, "rb") as f: 
            data = f.read()

        encryption_time, result = measure_performance_time(custom_encrypt, [public_key, data])
        ciphertext, hash_time = result
        total_time += encryption_time
                
        output_dir = save_in_results(ciphertext, folder_path, file_size)
        print(f"{file_name:<20} | {len(data):>10} | {hash_time:12.6f} | {encryption_time:12.6f}")
        print("-" * 65)
        
    print(f"Total time for folder {file_size}: {total_time:.6f} seconds")
    print(f"Files saved in: {output_dir}\n")
else: print("The folder is empty.")


Searching in: c:\code\S&P_TP1\SeP-TP1\03 - data\inputs by size
Available sizes: 8, 64, 512, 4096, 32768, 262144, 2097152

Processing 10 files from folder '8'...
File                 | Size       | Hash Time (s) | RSA Time (s)
-----------------------------------------------------------------
file_1.txt           |          8 |     0.000032 |     0.000448
-----------------------------------------------------------------
file_10.txt          |          8 |     0.000020 |     0.000368
-----------------------------------------------------------------
file_2.txt           |          8 |     0.000016 |     0.000339
-----------------------------------------------------------------
file_3.txt           |          8 |     0.000015 |     0.000342
-----------------------------------------------------------------
file_4.txt           |          8 |     0.000016 |     0.000352
-----------------------------------------------------------------
file_5.txt           |          8 |     0.000016 |     0.0